# Will lunch be good today? Logistic regression from scratch

This notebook builds a model that **learns** to predict whether the mess lunch will be good, using nothing but NumPy. No machine-learning library: every step, including the "learning", is written out so you can see it.

You need: what a function is, what a vector is, and the idea of a slope. That's it.

**The plan**
1. Look at the data
2. Split it into "study" and "exam" lunches
3. Put every feature on the same scale
4. Turn a score into a probability (sigmoid)
5. Measure how wrong the model is (loss)
6. Find which way is downhill (gradient)
7. Walk downhill, step by step (gradient descent)
8. Take the exam (test accuracy)
9. See what happens when the steps are too small or too big

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = "../../datasets/binary_classification.csv"   # path from this notebook's folder
TARGET = "good_lunch"

## 1. Look at the data

Each row is one logged lunch. `good_lunch` is the answer we want to predict: **1 = good, 0 = bad**. The other columns are the clues (called *features*).

In [ ]:
with open(CSV_PATH, newline="") as f:
    rows = list(csv.DictReader(f))

features = [c for c in rows[0] if c != TARGET]
X = np.array([[float(r[c]) for c in features] for r in rows])
y = np.array([float(r[TARGET]) for r in rows])

print("Features:", features)
print("Lunches:", len(y), "| share that were good:", round(y.mean(), 2))
print("First 3 lunches:")
for i in range(3):
    print("  ", dict(zip(features, X[i].tolist())), "-> good_lunch =", int(y[i]))

## 2. Split into "study" and "exam" lunches

If we test the model on the same lunches it learned from, it can just memorise them. That's like grading a student on the exact questions from their homework. So we hide **20%** of lunches as an exam (the *test set*) and only let the model learn from the other 80% (the *training set*).

We shuffle with a **fixed seed** so the split is the same every time anyone runs this.

In [ ]:
rng = np.random.default_rng(42)
order = rng.permutation(len(y))
n_test = len(y) // 5
test_idx, train_idx = order[:n_test], order[n_test:]

X_train_raw, X_test_raw = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print("Training lunches:", len(y_train), "| Exam lunches:", len(y_test))

## 3. Put every feature on the same scale

Features come in different units: one might be in the tens, another in the hundreds. Gradient descent takes one step size for *all* weights, so a feature with big numbers would dominate the steps (the same problem we saw with song duration in Level 1).

We **standardise**: subtract each feature's mean and divide by its spread, so every feature is centred on 0 with a spread of about 1. The mean and spread come from the **training** lunches only. The exam lunches must stay unseen.

In [ ]:
mean = X_train_raw.mean(axis=0)
std = X_train_raw.std(axis=0)
std[std == 0] = 1.0                       # a constant feature: leave it alone

X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std
print("Training feature means now:", X_train.mean(axis=0).round(3))
print("Training feature spreads now:", X_train.std(axis=0).round(3))

## 4. From a score to a probability

The model gives each feature a **weight** (how much it matters, and in which direction) and adds a **bias**:

$$z = w_1 x_1 + w_2 x_2 + \dots + b$$

$z$ can be any number, like 47 or −3. We want a probability between 0 and 1, so we squash $z$ with the **sigmoid**:

$$p = \frac{1}{1 + e^{-z}}$$

Big positive $z$ gives $p \approx 1$, big negative gives $p \approx 0$, and $z = 0$ gives exactly 0.5. We use the stable version from Level 1, so huge negative scores don't crash it.

In [ ]:
def sigmoid(z):
    e = np.exp(-np.abs(z))                     # only ever e^(negative) -> no overflow
    return np.where(z >= 0, 1 / (1 + e), e / (1 + e))

def predict_proba(X, w, b):
    return sigmoid(X @ w + b)                  # X @ w is a dot product for every lunch at once

for z in [-47, -2, 0, 2, 47]:
    print(f"z = {z:>4}  ->  p = {float(sigmoid(np.array(z))):.4f}")

## 5. How wrong is the model? (the loss)

We need one number that says how bad the current weights are. We use **cross-entropy** (also called log loss):

$$L = -\frac{1}{n}\sum \big[\, y \log p + (1-y)\log(1-p) \,\big]$$

In words: if the lunch was good ($y=1$), we pay $-\log p$. That's small when $p$ is near 1 and **huge** when we confidently said it would be bad. Being confidently wrong hurts much more than being unsure.

Starting with all weights at 0, every prediction is $p = 0.5$, and the loss is $\log 2 \approx 0.693$. That's our "knows nothing" starting point.

In [ ]:
def log_loss(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)           # avoid log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

w = np.zeros(X_train.shape[1])
b = 0.0
print("Loss with all-zero weights:", round(log_loss(predict_proba(X_train, w, b), y_train), 4))

## 6. Which way is downhill? (the gradient)

Picture the loss as a hilly landscape, where each position is one choice of weights. We want the lowest valley. The **gradient** is the slope in every direction: it points uphill, so we step the opposite way.

For this model, the calculus (chain rule through the sigmoid and the log) simplifies a lot:

$$\frac{\partial L}{\partial w} = \frac{1}{n} X^\top (p - y), \qquad \frac{\partial L}{\partial b} = \text{mean}(p - y)$$

$(p - y)$ is just **"how wrong was each prediction"**. Each weight's slope adds up those errors, weighted by how much that feature was present. A feature that is high whenever we're too optimistic gets pushed down.

In [ ]:
def gradients(X, y, w, b):
    error = predict_proba(X, w, b) - y         # positive = too optimistic, negative = too pessimistic
    return X.T @ error / len(y), error.mean()

gw, gb = gradients(X_train, y_train, w, b)
for name, g in zip(features, gw):
    print(f"slope for {name:<20} {g:+.4f}")
print(f"slope for bias{'':<16} {gb:+.4f}")

A **negative** slope means "increasing this weight lowers the loss", so that weight will grow. Compare the signs with your intuition about each feature.

## 7. Walk downhill (gradient descent)

Repeat: compute the gradient, take a small step against it.

$$w \leftarrow w - \eta \, \frac{\partial L}{\partial w}$$

$\eta$ (eta) is the **learning rate**, the size of each step. We stop when the ground is flat: the length of the gradient falls below $10^{-4}$. That's our **convergence criterion**.

In [ ]:
def train(X, y, lr, max_epochs=20000, tol=1e-4):
    w, b = np.zeros(X.shape[1]), 0.0
    losses = []
    for epoch in range(max_epochs):
        losses.append(log_loss(predict_proba(X, w, b), y))
        gw, gb = gradients(X, y, w, b)
        if np.sqrt(np.sum(gw**2) + gb**2) < tol:     # flat enough: converged
            return w, b, losses, True
        w = w - lr * gw                               # the actual learning
        b = b - lr * gb
    return w, b, losses, False

w, b, losses, converged = train(X_train, y_train, lr=0.1)
print(f"Converged: {converged} after {len(losses)} steps")
print(f"Loss went from {losses[0]:.4f} to {losses[-1]:.4f}")
print("Learned weights:")
for name, wi in zip(features, w):
    print(f"  {name:<20} {wi:+.3f}")

Because the features are standardised, the weights can be compared directly: a bigger number (ignoring sign) means a stronger effect, and the sign says whether the feature makes a good lunch more or less likely.

## 8. Take the exam

Predict "good" when $p \ge 0.5$, and check against the hidden exam lunches. A model is only useful if it beats the lazy strategy of **always guessing the most common answer**.

In [ ]:
def accuracy(X, y, w, b):
    return np.mean((predict_proba(X, w, b) >= 0.5) == y)

baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"Lazy baseline: {baseline:.1%}")
print(f"Our model on the exam lunches: {accuracy(X_test, y_test, w, b):.1%}")
print(f"Our model on the study lunches: {accuracy(X_train, y_train, w, b):.1%}")

If the study score is much higher than the exam score, the model memorised instead of learning. Close scores mean it learned something general.

## 9. Step size matters

Try three learning rates. **Guess first:** what will a tiny step (0.001) and a huge step (50) do compared with 0.1?

In [ ]:
plt.figure(figsize=(8, 5))
for lr in [0.001, 0.1, 50]:
    _, _, l, ok = train(X_train, y_train, lr)
    plt.plot(range(1, len(l) + 1), l, label=f"lr = {lr}" + ("" if ok else " (not converged)"))
plt.xscale("log")
plt.xlabel("Step (log scale)")
plt.ylabel("Training loss")
plt.title("Too small, about right, too big")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

- **Too small (0.001):** every step is safe but tiny, so it's still walking after 20,000 steps.
- **About right (0.1):** a smooth slide into the valley, and it stops when the ground is flat.
- **Too big (50):** each step jumps *over* the valley to the other side, sometimes landing higher than where it started. It bounces back and forth and never settles.

## Recap

| Step | Idea | One line of code |
|---|---|---|
| Score | weighted sum of features | `X @ w + b` |
| Probability | squash with sigmoid | `sigmoid(z)` |
| Loss | punish confident mistakes | `-mean(y log p + (1-y) log(1-p))` |
| Gradient | how wrong × how present | `X.T @ (p - y) / n` |
| Learning | step downhill | `w -= lr * gradient` |

That's the whole of logistic regression. The same loop of predicting, measuring the error, finding the slope and stepping is how much bigger neural networks learn, too.